# Path B (time-domain latent): class-conditional diffusion over DAC's continuous latent

Diffusion over **time-domain codec tokens**, done the way a diffusion model can learn it. A pretrained
neural codec (DAC, 16 kHz) encodes each 2-second crop into 104 frames (one every 20 ms). Each frame's
continuous latent, the 1,024-dim vector DAC's decoder reads, is reduced with PCA to `LATENT_DIM`
numbers. The diffusion model generates those, and DAC's decoder turns them back into audio. No
spectrogram is involved.

**Why not the token-embedding version** (`path_b_codec_diffusion.ipynb`)? That run diffused the 12
codebooks' raw vectors stacked together. The diagnostic showed the model barely learned it: v-loss of
0.76 on the first codebook and about 0.95 on codebooks 4 to 12, where 1.0 means nothing was learned.
Raw codebook vectors are not arranged so that nearby vectors sound alike, and the fine codebooks look
like noise but carried equal weight in the loss. DAC's continuous latent is smooth by construction
(its decoder maps nearby latents to similar audio), and PCA orders its directions by how much audio
they carry. The token-embedding run is kept as a reported negative result.

Data, split, training loop, and sampler come from `common.py`, as in every other path.

Set `QUICK_TEST = True` first: it runs in a few minutes and prints the table you use to choose
`LATENT_DIM`.

## 1. Setup

In [ ]:
!pip install -q diffusers accelerate soundfile "transformers>=4.45"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, sys, math, time
REPO_DIR = "/content/thesis"
if not os.path.exists(REPO_DIR):
    !git clone -q https://github.com/1jvaldez1/audio-diffusion-synth.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q
sys.path.insert(0, REPO_DIR)
import common as C
import subprocess

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import Audio, display

assert hasattr(C, "LatentFrontEnd"), "common.py is an old version: pull the repo, then restart the session"
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

## 2. Config

`LATENT_DIM` is the one new setting. Choose it from the table in section 3: the smallest value whose
round-trip error is close to the codec's own (about 2 dB on ESC-50). `total_steps` stays at the
default (10,000), the same budget as the other paths.

In [ ]:
QUICK_TEST = False          # True: short run to check the notebook end to end

cfg = C.Config(total_steps=300, inference_steps=20) if QUICK_TEST else C.Config()
C.seed_everything(cfg.seed)

# Path B-latent settings
LATENT_DIM  = 64            # PCA directions kept per frame; choose from the table in section 3
WHITEN      = False         # False: one global scale (keeps PCA's importance order); True: ablation
N_CODEBOOKS = 12            # tokens used to form the latent; 12 = DAC's full quality
CHANNELS    = (128, 256, 256)
ATTN_LEVELS = (0, 1, 2)
DROPOUT     = 0.1
GUIDANCE    = 1.0           # keep at 1.0 for distribution metrics
N_PER_CLASS = 2 if QUICK_TEST else 16

TAG          = f"_k{LATENT_DIM}" + ("_white" if WHITEN else "") + ("_quick" if QUICK_TEST else "")
RUN_DIR      = f"{cfg.out_root}/path_b_latent{TAG}"
MEL_FRONTEND = f"{cfg.out_root}/frontend.json"                        # shared; created by the Path A notebook
TOKEN_CACHE  = f"{cfg.out_root}/tokens_train_{N_CODEBOOKS}cb.pt"       # every training clip's tokens, encoded once
LATENT_FILE  = f"{cfg.out_root}/frontend_latent_{N_CODEBOOKS}cb.pt"    # the PCA (all directions; LATENT_DIM picks how many)
WAVE_FILE    = f"{cfg.out_root}/samples/path_b_latent_wave{TAG}.pt"    # generated audio
MEL_FILE     = f"{cfg.out_root}/samples/path_b_latent{TAG}.pt"         # the same audio as mel
TOKENS_FILE  = f"{cfg.out_root}/samples/path_b_latent_tokens{TAG}.pt"  # the generated latents re-quantized to DAC tokens

## 3. Data, tokens, and the latent front end

1. **Tokens, once.** Every training clip is encoded whole by DAC a single time and cached on Drive
   (shared with the token-embedding notebook if it already made the file). Training cuts fresh random
   104-frame windows from the cache, so crop augmentation is kept without running the encoder each step.
2. **Latent.** A window's tokens are turned into DAC's continuous latent: each codebook's vector is
   projected back to 1,024 dims and the 12 are summed. This is exactly what DAC's decoder reads, so
   it is a table lookup plus a small linear map, not a network pass.
3. **PCA.** Fitted once on frames from 8 random training windows per clip (training data only), and
   saved. Every direction is stored, so `LATENT_DIM` can be changed later without refitting.

In [ ]:
data = C.load_esc50(cfg)
train_split, val_split, heldout_split = C.split_by_fold(data, cfg)
print(f"train clips: {len(train_split['waveforms'])} | val clips: {len(val_split['waveforms'])} | held-out clips: {len(heldout_split['waveforms'])}")

assert os.path.exists(MEL_FRONTEND), "run the Path A notebook first: it creates the shared mel front end"
mel_frontend = C.MelFrontEnd.load(MEL_FRONTEND, cfg, device=device)
codec = C.CodecFrontEnd(cfg, n_codebooks=N_CODEBOOKS, device=device)

if os.path.exists(TOKEN_CACHE):
    train_tokens = torch.load(TOKEN_CACHE)
else:
    t0 = time.time()
    train_tokens = codec.encode_clips(train_split["waveforms"])
    torch.save(train_tokens, TOKEN_CACHE)
    print(f"encoded {len(train_tokens)} training clips once in {time.time() - t0:.0f} s")
assert len(train_tokens) == len(train_split["waveforms"]), f"{TOKEN_CACHE} does not match the training split"
train_set = C.TokenCropDataset(train_split, codec, cfg, tokens=train_tokens)

if os.path.exists(LATENT_FILE):
    latent_fe = C.LatentFrontEnd.load(LATENT_FILE, codec, k=LATENT_DIM, whiten=WHITEN)
else:
    torch.manual_seed(cfg.seed)
    windows = [train_set[i % len(train_set)][0] for i in range(8 * len(train_set))]
    latent_fe = C.LatentFrontEnd(codec, k=LATENT_DIM, whiten=WHITEN).fit(windows)
    latent_fe.save(LATENT_FILE)

SAMPLE_SHAPE = (LATENT_DIM, codec.frames)
x = latent_fe.from_tokens(torch.stack([train_set[i % len(train_set)][0] for i in range(64)]))
print(f"model input {SAMPLE_SHAPE[0]} x {SAMPLE_SHAPE[1]} = {SAMPLE_SHAPE[0] * SAMPLE_SHAPE[1]:,} values per clip "
      f"(raw Path B: {cfg.clip_samples:,}; token embeddings: {codec.channels * codec.frames:,})")
print(f"PCA keeps {latent_fe.explained():.1%} of the latent's variance | input mean {x.mean():.3f}, "
      f"average variance {x.var(dim=(0, 2)).mean():.3f} (about 1)")

### Choosing `LATENT_DIM`: what the PCA costs

Validation crops pushed through codec + PCA + codec decoder at several sizes, compared with the
original audio in mel space. The `codec only` row keeps every direction: that is the codec's own
error, the floor. Pick the smallest size within a few tenths of a dB of that floor, then listen. A
smaller latent is easier to model, so the smallest transparent size is the right one.

In [ ]:
ref_wave, ref_labels = C.reference_crops(val_split, cfg, k=1)
probe = ref_wave[:64]
to_mel = lambda w: torch.cat([mel_frontend(w[i : i + 64]).cpu() for i in range(0, len(w), 64)])
probe_mel = to_mel(probe)

print(f"{'directions':>12}{'variance kept':>16}{'mel error (dB)':>16}{'95th pct':>10}")
rows = {}
for k in (8, 16, 32, 64, 128, 256, 1024):
    latent_fe.set_k(k)
    e = (to_mel(latent_fe.roundtrip(probe)) - probe_mel).abs() * cfg.db_range / 2
    rows[k] = e.mean().item()
    label = f"{k}" + (" (codec only)" if k == 1024 else "")
    print(f"{label:>12}{latent_fe.explained():>16.1%}{rows[k]:>16.2f}{e.flatten()[::97].quantile(0.95).item():>10.2f}")
latent_fe.set_k(LATENT_DIM)

plt.figure(figsize=(5, 3))
plt.semilogy(latent_fe.eigvals.cpu().numpy()[:256])
plt.xlabel("PCA direction"); plt.ylabel("variance"); plt.title("How DAC's latent variance is spread")
plt.show()

rt = latent_fe.roundtrip(probe[:2])
for i in range(2):
    print(data["class_names"][ref_labels[i]], f"- original, then codec + PCA at LATENT_DIM = {LATENT_DIM}")
    display(Audio(probe[i, 0].numpy(), rate=cfg.sample_rate))
    display(Audio(rt[i, 0].numpy(), rate=cfg.sample_rate))

## 4. Model

The same 1D UNet as the token-embedding notebook, with `LATENT_DIM` input channels. Every choice
and the reason for it:

| Choice | Reason |
|---|---|
| Diffuse DAC's **continuous** latent, not its token vectors | Gaussian diffusion learns by averaging: early in sampling it predicts the mean of plausible outputs and refines from there. That only works if averages of valid inputs are themselves meaningful. DAC's latent is what its decoder reads, and the decoder maps nearby latents to similar audio, so averages decode to plausible blurred audio that later steps sharpen. Raw codebook vectors have no such geometry, which is why the token-embedding run failed. |
| No snapping to tokens | The decoder takes the continuous latent directly, so generation never makes a hard nearest-neighbour choice that a small error could flip. Tokens are still produced afterwards, by re-quantizing the generated latent with DAC's own quantizer, for the "tokens in the time domain" framing and later work. |
| PCA to `LATENT_DIM` directions, fitted on training frames only | DAC's latent has 1,024 dims per frame but its variance is concentrated in far fewer directions (see the plot in section 3). Keeping only those makes the model's job smaller with a measured, audible-or-not cost (the table in section 3). PCA is linear, so it keeps the latent's smoothness. |
| One global scale, not per-direction whitening | Diffusion schedules assume data with an overall spread near 1, so the coefficients are divided by a single number. Per-direction whitening would also equalise every direction's weight in the loss, giving a direction that carries almost no audio as much weight as the dominant ones. That is the failure seen in the token-embedding run. With one global scale the loss is largest where the audio is. `WHITEN = True` runs the ablation. |
| Same ResNet block, class embedding with a "no label" entry, v-prediction, schedule, EMA, sampler as every other path | The comparison should isolate the representation. |
| Three levels (128 / 256 / 256), 104 -> 52 -> 26 positions | 104 = 8 x 13 halves cleanly at most three times; two halvings give a 26-position bottleneck of about 80 ms per position without blurring onsets. |
| Self-attention at every level | The longest sequence is 104 positions, so global attention is cheap everywhere, and with 20 ms frames and kernel-3 convolutions (60 ms reach) attention is what lets each frame see the whole clip. `ATTN_LEVELS = (2,)` is the strict ablation. |
| About 15.4M parameters | Matched to Path A (16.4M) and raw Path B (15.9M). `LATENT_DIM` changes only the input and output convolutions. |
| Kernel 3, nearest-neighbour upsampling then a conv | Mirrors Paths A and B; avoids transposed-convolution checkerboard artefacts. |

In [ ]:
def _norm(ch):
    return nn.GroupNorm(min(32, ch // 4), ch)


class ResBlock1D(nn.Module):
    """GroupNorm -> SiLU -> conv, twice, with the conditioning applied as scale and shift."""

    def __init__(self, in_ch, out_ch, emb_dim, dropout, kernel):
        super().__init__()
        self.norm1, self.conv1 = _norm(in_ch), nn.Conv1d(in_ch, out_ch, kernel, padding=kernel // 2)
        self.emb = nn.Linear(emb_dim, 2 * out_ch)
        self.norm2, self.conv2 = _norm(out_ch), nn.Conv1d(out_ch, out_ch, kernel, padding=kernel // 2)
        self.dropout = nn.Dropout(dropout)
        self.skip = nn.Conv1d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()

    def forward(self, x, emb):
        h = self.conv1(F.silu(self.norm1(x)))
        scale, shift = self.emb(F.silu(emb)).unsqueeze(-1).chunk(2, dim=1)
        h = self.norm2(h) * (1 + scale) + shift
        h = self.conv2(self.dropout(F.silu(h)))
        return self.skip(x) + h


class AttnBlock1D(nn.Module):
    """Self-attention over token positions, with a residual connection."""

    def __init__(self, ch, head_dim):
        super().__init__()
        self.norm = _norm(ch)
        self.attn = nn.MultiheadAttention(ch, ch // head_dim, batch_first=True)

    def forward(self, x):
        h = self.norm(x).transpose(1, 2)                      # (B, T, C)
        h, _ = self.attn(h, h, h, need_weights=False)
        return x + h.transpose(1, 2)


class TokenUNet1D(nn.Module):
    """Path B's 1D UNet adapted to a sequence of token embeddings.

    Same ResBlock / attention / conditioning design as Path B. Differences: the
    input already has `in_channels` features per position (no sample folding),
    and attention runs at every level listed in `attn_levels`.
    """

    def __init__(self, num_classes, in_channels=96, channels=(256, 384, 512), attn_levels=(1, 2),
                 layers_per_block=2, dropout=0.1, head_dim=64, kernel=3):
        super().__init__()
        self.base, emb_dim = channels[0], channels[0] * 4
        self.time_mlp = nn.Sequential(nn.Linear(self.base, emb_dim), nn.SiLU(), nn.Linear(emb_dim, emb_dim))
        self.class_emb = nn.Embedding(num_classes + 1, emb_dim)          # +1 = "no label"
        self.conv_in = nn.Conv1d(in_channels, channels[0], kernel, padding=kernel // 2)

        def res(i, o):
            return ResBlock1D(i, o, emb_dim, dropout, kernel)

        def attn(lvl, ch):
            return AttnBlock1D(ch, head_dim) if lvl in attn_levels else nn.Identity()

        last = len(channels) - 1
        self.down, skip_ch, ch = nn.ModuleList(), [channels[0]], channels[0]
        for lvl, out in enumerate(channels):
            for _ in range(layers_per_block):
                self.down.append(nn.ModuleList([res(ch, out), attn(lvl, out)]))
                ch = out; skip_ch.append(ch)
            if lvl != last:
                self.down.append(nn.Conv1d(ch, ch, 3, stride=2, padding=1)); skip_ch.append(ch)

        self.mid = nn.ModuleList([res(ch, ch), AttnBlock1D(ch, head_dim), res(ch, ch)])

        self.up = nn.ModuleList()
        for lvl, out in reversed(list(enumerate(channels))):
            for _ in range(layers_per_block + 1):
                self.up.append(nn.ModuleList([res(ch + skip_ch.pop(), out), attn(lvl, out)]))
                ch = out
            if lvl != 0:
                self.up.append(nn.Conv1d(ch, ch, 3, padding=1))            # applied after 2x nearest upsampling

        self.norm_out, self.conv_out = _norm(ch), nn.Conv1d(ch, in_channels, kernel, padding=kernel // 2)

    def forward(self, x, t, labels):
        half = self.base // 2
        freqs = torch.exp(-math.log(10000.0) * torch.arange(half, device=x.device) / half)
        args = t.float()[:, None] * freqs[None]
        emb = self.time_mlp(torch.cat([args.sin(), args.cos()], dim=1).to(x.dtype)) + self.class_emb(labels)

        h = self.conv_in(x)
        skips = [h]
        for m in self.down:
            if isinstance(m, nn.ModuleList):
                h = m[1](m[0](h, emb))
            else:
                h = m(h)
            skips.append(h)
        h = self.mid[2](self.mid[1](self.mid[0](h, emb)), emb)
        for m in self.up:
            if isinstance(m, nn.ModuleList):
                h = m[1](m[0](torch.cat([h, skips.pop()], dim=1), emb))
            else:
                h = m(F.interpolate(h, scale_factor=2.0, mode="nearest"))
        return self.conv_out(F.silu(self.norm_out(h)))


def forward_fn(model, x, t, labels):
    return model(x, t, labels)

unet = TokenUNet1D(cfg.num_classes, in_channels=LATENT_DIM, channels=CHANNELS,
                   attn_levels=ATTN_LEVELS, dropout=DROPOUT).to(device)
n_params = sum(p.numel() for p in unet.parameters())
print(f"Path B-latent params: {n_params:,}")

## 5. Train

Resumes from `RUN_DIR/last.pt` if it exists. Each step is a token lookup, a linear map, and the
diffusion model; the codec encoder ran once above. Record the training time next to raw Path B's: it
is the efficiency result. For comparison, the token-embedding run's loss flattened near 0.93.

In [ ]:
t0 = time.time()
ema, losses = C.train_diffusion(unet, forward_fn, latent_fe.from_tokens, train_set, cfg, RUN_DIR, device)
print(f"training time this session: {(time.time() - t0) / 60:.1f} min")

plt.figure(figsize=(5, 3))
plt.plot([cfg.log_every * (i + 1) for i in range(len(losses))], losses)
plt.xlabel("step"); plt.ylabel("loss"); plt.title("Path B-latent training loss")
plt.show()

### What the model learned, direction by direction

The same check that diagnosed the token-embedding run: v-prediction loss on training windows,
averaged over noise levels, grouped by PCA direction. Because the directions have different
variances, each is shown relative to the loss of a model that learned nothing beyond each direction's
variance (the best guess from the noisy input alone, a linear rescaling). 1.0 means nothing was learned,
exactly as in the token-embedding check, so the two runs can be compared directly. Expect the leading
directions, which carry most of the audio, to be learned best.

In [ ]:
scheduler, _ = C.make_schedulers(cfg)
ema.store(unet.parameters()); ema.copy_to(unet.parameters()); unet.eval()
batch = [train_set[i % len(train_set)] for i in range(256)]
x0 = latent_fe.from_tokens(torch.stack([b[0] for b in batch]))
labels = torch.stack([b[1] for b in batch]).to(device)
var = x0.var(dim=(0, 2)).cpu()
err, base = torch.zeros(LATENT_DIM), torch.zeros(LATENT_DIM)
with torch.no_grad():
    for t_val in (100, 300, 500, 700, 900):
        t = torch.full((len(x0),), t_val, device=device)
        noise = torch.randn_like(x0)
        pred = unet(scheduler.add_noise(x0, noise, t), t, labels)
        err += ((pred - scheduler.get_velocity(x0, noise, t)) ** 2).mean(dim=(0, 2)).cpu() / 5
        # Loss of the best "learned nothing" predictor: a linear rescaling of the noisy input, per direction.
        ac = scheduler.alphas_cumprod[t_val].item(); a, b = ac ** 0.5, (1 - ac) ** 0.5
        base += ((a**2 + b**2 * var) - (a * b * (1 - var)) ** 2 / (a**2 * var + b**2)) / 5
ema.restore(unet.parameters()); unet.train()
rel = err / base
edges = [0, 4, 8, 16, 32, 64, 128, 256]
for lo, hi in zip(edges[:-1], edges[1:]):
    if lo < LATENT_DIM:
        hi = min(hi, LATENT_DIM)
        print(f"directions {lo + 1:>3}-{hi:<3} relative v-loss {rel[lo:hi].mean():.3f}   (data variance {var[lo:hi].mean():.3f})")
print(f"variance-weighted relative v-loss: {(rel * var).sum() / var.sum():.3f}   (token-embedding run: about 0.93)")

## 6. Generate and save samples

Same file formats as the other paths. `WAVE_FILE` is what the evaluation notebook reads.

In [ ]:
commit = subprocess.check_output(["git", "-C", REPO_DIR, "rev-parse", "--short", "HEAD"]).decode().strip()

gen_labels = C.balanced_labels(cfg, N_PER_CLASS)
generated = C.sample(unet, forward_fn, SAMPLE_SHAPE, cfg, device,
                     labels=gen_labels, ema=ema, guidance=GUIDANCE)
gen_wave, gen_tokens = latent_fe.to_wave(generated)
gen_mel = to_mel(gen_wave)

meta = dict(path="B_latent", guidance=GUIDANCE, steps=cfg.inference_steps, codec=C.CodecFrontEnd.MODEL_ID,
            n_codebooks=N_CODEBOOKS, latent_dim=LATENT_DIM, whiten=WHITEN, cache_tokens=True,
            pca_variance_kept=latent_fe.explained(), channels=CHANNELS, attn_levels=ATTN_LEVELS,
            n_params=n_params, train_steps=cfg.total_steps, commit=commit)
C.save_samples(WAVE_FILE, gen_wave, gen_labels, cfg, representation="waveform", **meta)
C.save_samples(MEL_FILE, gen_mel, gen_labels, cfg, representation="mel", **meta)
C.save_samples(TOKENS_FILE, gen_tokens, gen_labels, cfg, representation="dac_tokens", **meta)
print("saved", tuple(gen_wave.shape), tuple(gen_mel.shape), tuple(gen_tokens.shape), "| code version:", commit)

# Do generated latents have the spread of real ones, direction by direction?
real_x = latent_fe.from_tokens(torch.stack([train_set[i % len(train_set)][0] for i in range(len(generated))])).cpu()
ratio = generated.std(dim=(0, 2)) / real_x.std(dim=(0, 2))
print(f"generated / real std per direction: first 8 {[round(v, 2) for v in ratio[:8].tolist()]}, median {ratio.median():.2f}")
crop_set = C.CropDataset(train_split, cfg)
real_w = torch.stack([crop_set[i % len(crop_set)][0] for i in range(64)])
print(f"waveform std {gen_wave.std():.3f} (real: {real_w.std():.3f})")

## 7. Look and listen

Held-out real clips (top) against generated ones of the same class (bottom), then audio players.
A sanity check only; the evaluation notebook does the scoring.

In [ ]:
ref_mel = to_mel(ref_wave)
show = torch.randperm(cfg.num_classes)[:5].tolist()

fig, axes = plt.subplots(2, 5, figsize=(16, 5))
for col, c in enumerate(show):
    real = ref_mel[ref_labels == c][0, 0]
    fake = gen_mel[gen_labels == c][0, 0]
    for row, (img, tag) in enumerate([(real, "real"), (fake, "generated")]):
        axes[row, col].imshow(img, origin="lower", aspect="auto", cmap="magma", vmin=-1, vmax=1)
        axes[row, col].set_title(f"{data['class_names'][c]} ({tag})")
        axes[row, col].set_xticks([]); axes[row, col].set_yticks([])
plt.tight_layout(); plt.show()

for c in show[:3]:
    print(data["class_names"][c], "- real, then two generated")
    display(Audio(ref_wave[ref_labels == c][0, 0].numpy(), rate=cfg.sample_rate))
    for j in range(2):
        display(Audio(gen_wave[gen_labels == c][j, 0].numpy(), rate=cfg.sample_rate))